In [ ]:
!pip install transformers sentence-transformers torch accelerate datasets scikit-learn


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline, AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer, util
import pandas as pd
import numpy as np


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ipc_df = pd.read_csv("/content/drive/MyDrive/ipc_clean.csv")
crpc_df = pd.read_csv("/content/drive/MyDrive/crpc_clean.csv")
constitution_df = pd.read_csv("/content/drive/MyDrive/constitution_articles_cleaned.csv")

ipc_df["domain"] = "IPC"
crpc_df["domain"] = "CrPC"
constitution_df["domain"] = "Constitution"

df = pd.concat([ipc_df, crpc_df, constitution_df], ignore_index=True)
df = df.drop_duplicates(subset=["question"]).dropna(subset=["question", "answer"])

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch

# Encode labels
le = LabelEncoder()
df["label"] = le.fit_transform(df["domain"])

# Create dataset
dataset = Dataset.from_pandas(df[["question", "label"]])

# Tokenizer
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(
        examples["question"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized = dataset.map(tokenize_function, batched=True)
train_test = tokenized.train_test_split(test_size=0.2)

# Model
bert_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(le.classes_)
)

# Compute class weights
classes = np.unique(df["label"])
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=df["label"]
)
class_weights = torch.tensor(class_weights, dtype=torch.float)

# Custom Trainer with weighted loss
from transformers import Trainer

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights.to(model.device))
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# Training settings
training_args = TrainingArguments(
    output_dir="./bert_intent_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
)

# Trainer
trainer = WeightedTrainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_test["train"],
    eval_dataset=train_test["test"],
    processing_class=tokenizer,
)

# Train model
trainer.train()


In [ ]:
trainer.save_model("./bert_intent_model")
tokenizer.save_pretrained("./bert_intent_model")

intent_classifier = pipeline("text-classification", model="./bert_intent_model", tokenizer="./bert_intent_model", return_all_scores=True)


In [ ]:
# Load one model at a time due to Colab VRAM limits
def load_llama_model(model_path="TinyLlama/TinyLlama-1.1B-Chat-v1.0"):
    print(f"Loading model: {model_path}")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map="auto",
        torch_dtype=torch.float16,
    )
    return tokenizer, model



In [ ]:
def generate_answer(question, predicted_domain):
    # Select dataset based on predicted domain
    if predicted_domain == "Constitution":
        data = constitution_df
    elif predicted_domain == "CrPC":
        data = crpc_df
    else:
        data = ipc_df

    # Semantic search for context
    embedder = SentenceTransformer("all-MiniLM-L6-v2")
    q_emb = embedder.encode(question, convert_to_tensor=True)
    all_embs = embedder.encode(data["question"].tolist(), convert_to_tensor=True)

    sims = util.cos_sim(q_emb, all_embs)[0]
    best_idx = torch.argmax(sims).item()
    context = data.iloc[best_idx]["answer"]
    score = sims[best_idx].item()

    # Load lightweight LLaMA model
    llama_tokenizer, llama_model = load_llama_model("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

    # Prompt (concise, friendly, avoids repetition)
    prompt = f"""
You are an AI legal assistant.
Answer the following question using the context provided.
Use simple, human-like language and keep it under 4 sentences.
Avoid repeating phrases or rephrasing the same point.

Question: {question}
Relevant legal context: {context}
Answer:
"""

    inputs = llama_tokenizer(prompt, return_tensors="pt").to("cuda")
    output = llama_model.generate(
        **inputs,
        max_new_tokens=120,      # shorter answers
        temperature=0.5,         # more focused, less randomness
        top_p=0.9,               # nucleus sampling
        repetition_penalty=1.8,  # discourages repetition
        do_sample=True
    )

    answer = llama_tokenizer.decode(output[0], skip_special_tokens=True)

    # Extract only the assistant’s answer (removes repeated prompt)
    if "Answer:" in answer:
        answer = answer.split("Answer:")[-1].strip()

    return answer, score


In [ ]:
# Function to ask one question
def ask_chatbot(question):
    # Step 1: Predict domain
    preds = intent_classifier(question)

    top_pred = max(preds, key=lambda x: x["score"])

    # Extract the numerical part from the label (e.g., 'LABEL_0' → 0)
    # The 'label' key is guaranteed to exist now in top_pred.
    predicted_label_index = int(top_pred["label"].split('_')[-1])
    predicted_domain = le.inverse_transform([predicted_label_index])[0]

    print(f"\n Predicted Domain: {predicted_domain}")

    # Step 2: Generate answer
    answer, similarity_score = generate_answer(question, predicted_domain)
    print(f"\nAnswer (Similarity Score: {similarity_score:.2f}):\n{answer}")
    print("-" * 90)


# Sample questions for testing
test_questions = [
    # Constitution
    "What is the right to equality under the Constitution?",
    "What are the fundamental duties mentioned in the Indian Constitution?",

    # IPC (Indian Penal Code)
    "What is the punishment for theft under IPC?",
    "What does Section 300 of the IPC define?",

    # CrPC (Criminal Procedure Code)
    "What is the procedure for arrest without warrant under CrPC?",
    "What are the powers of a magistrate under CrPC?"
]


# Run all sample questions
for q in test_questions:
    ask_chatbot(q)
